# Part B (optional extension, prepared, NOT run): a larger 4-class MRI dataset on Colab

Dataset: Kaggle "Brain Tumor MRI Dataset" by Masoud Nickparvar (about 7,000 images; glioma, meningioma, pituitary, no tumour).

**Not a medical tool.** This is a parallel-computing and evaluation exercise. Nothing here is fit for diagnosis.

Status: prepared, **not executed** by me. No results from it are claimed anywhere in this repo.

Before running: open the dataset page, read its licence and terms yourself, and only then set `LICENCE_CONFIRMED = True` in the next cell. Kaggle credentials must come from Colab secrets or environment variables; never write them into this notebook or any file in the repo.

In [ ]:
# Licence gate: this cell stops the notebook until you have checked the licence yourself.
LICENCE_CONFIRMED = False  # set to True only after reading the dataset licence on Kaggle
DATASET = "masoudnickparvar/brain-tumor-mri-dataset"
print(f"Dataset: https://www.kaggle.com/datasets/{DATASET}")
print(
    "Licence note: check the licence shown on the dataset page and whether it allows your use. "
    "Do not redistribute the images; do not commit them to any repository."
)
if not LICENCE_CONFIRMED:
    raise SystemExit(
        "STOP: licence not confirmed. Read the licence, then set LICENCE_CONFIRMED = True."
    )

## 1. Setup (Colab)

In [ ]:
# Install mri_pipeline. Works whether this repo is public or private.
# No credential is stored in this notebook, in the repo, or on disk.
import base64
import os
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = "maryam843/parallel-mri-image-pipeline"
CLONE_DIR = Path("/content/parallel-mri-image-pipeline")


def _pip_install(*args, env=None):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    return subprocess.run(cmd, capture_output=True, text=True, env=env).returncode == 0


def _local_copy():
    """A copy of the repo uploaded to Colab or already cloned (or this checkout)."""
    for folder in (Path.cwd(), Path.cwd().parent, CLONE_DIR):
        if (folder / "pyproject.toml").is_file() and (folder / "src" / "mri_pipeline").is_dir():
            return folder
    return None


def _colab_secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def install_mri_pipeline():
    # (a) local copy
    local = _local_copy()
    if local is not None and _pip_install("-e", str(local)):
        return "(a) installed from a local copy of the repo"
    # (b) public GitHub repo; never prompt for a password
    no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if _pip_install(f"git+https://github.com/{REPO}.git", env=no_prompt):
        return "(b) installed from the public GitHub repo"
    # (c) private repo with a Colab secret named GITHUB_TOKEN. The token is passed
    # to git only through environment variables for this one command, so it never
    # appears in a URL, in .git/config, in the output or in any file.
    token = _colab_secret("GITHUB_TOKEN")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env = {
            **no_prompt,
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic}",
        }
        del token, basic
        target = CLONE_DIR if CLONE_DIR.parent.is_dir() else Path(tempfile.mkdtemp()) / "repo"
        clone = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(target)],
            capture_output=True,
            text=True,
            env=env,
        )
        del env
        if clone.returncode == 0 and _pip_install("-e", str(target)):
            return "(c) installed from a private clone using the GITHUB_TOKEN Colab secret"
    # (d) nothing worked
    raise SystemExit(
        "Could not install mri_pipeline. Choose one option and run this cell again:\n"
        "  (a) upload or clone the repo into Colab, for example to\n"
        "      /content/parallel-mri-image-pipeline;\n"
        "  (b) make the GitHub repo public, then the pip install from GitHub works;\n"
        "  (c) add a Colab secret named GITHUB_TOKEN (a read-only token for this repo) "
        "in the Colab 'Secrets' panel and allow this notebook to use it."
    )


print(install_mri_pipeline())

In [ ]:
# Extra tools used only in this notebook, and Kaggle credentials from Colab secrets.
!pip -q install kagglehub torch torchvision
import os  # noqa: F811 - kept so this cell also runs on its own

try:
    from google.colab import userdata  # Colab secrets: KAGGLE_USERNAME, KAGGLE_KEY

    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    print("Not on Colab or no secrets set; kagglehub will use its own login flow.")

In [ ]:
from pathlib import Path

import kagglehub

root = Path(kagglehub.dataset_download(DATASET))
splits = {s: root / s for s in ("Training", "Testing")}
classes = sorted(p.name for p in splits["Training"].iterdir() if p.is_dir())
print("classes:", classes)

## 2. Exact and near-duplicate check across the train/test split

Exact duplicates: SHA-256 of the file bytes. Near duplicates: a 64-bit difference hash (dHash) of the image resized to 9x8, compared by Hamming distance. Any group that spans train and test is reported and removed from the test side before evaluation.

In [ ]:
import numpy as np
import pandas as pd

from mri_pipeline import io


def dhash(image, size=8):
    import cv2

    small = cv2.resize(image, (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = (small[:, 1:] > small[:, :-1]).ravel()
    return int("".join("1" if b else "0" for b in bits), 2)


rows = []
for split, folder in splits.items():
    recs = io.list_images(folder, classes={c: i for i, c in enumerate(classes)})
    for r in recs:
        rows.append(
            {
                "split": split,
                "class": r.class_name,
                "label": r.label,
                "path": r.path,
                "sha256": r.sha256,
                "dhash": dhash(io.load_image(r.path)),
            }
        )
df = pd.DataFrame(rows)
print(df.groupby(["split", "class"]).size().unstack())

exact_cross = set(df[df.split == "Training"].sha256) & set(df[df.split == "Testing"].sha256)
print("exact duplicates across splits:", len(exact_cross))

train_h = df[df.split == "Training"].dhash.to_numpy(dtype=np.uint64)


def near(h, max_bits=4):
    d = np.array([bin(int(x) ^ h).count("1") for x in train_h])
    return (d <= max_bits).any()


df_test = df[df.split == "Testing"].copy()
df_test["near_dup_in_train"] = [near(h) for h in df_test.dhash]
print(
    "test images with a near duplicate (<= 4 bits) in training:",
    int(df_test.near_dup_in_train.sum()),
)
clean_test = df_test[~df_test.near_dup_in_train & ~df_test.sha256.isin(exact_cross)]
print("clean test images kept:", len(clean_test))

## 3. Same parallel filter pipeline, speedup table on a larger subset

In [ ]:
import functools

from mri_pipeline.filters import apply_filters
from mri_pipeline.parallel import available_cores, timed_run

subset = df[df.split == "Training"].sample(n=200, random_state=0)
images = [io.load_image(p, size=256) for p in subset.path]
task = functools.partial(apply_filters, mode="fast")
t_seq = timed_run(task, images, strategy="sequential").seconds
table = [{"processes": 1, "seconds": t_seq, "speedup": 1.0}]
for p in range(2, available_cores() + 1):
    t = timed_run(task, images, strategy="map", processes=p).seconds
    table.append({"processes": p, "seconds": t, "speedup": t_seq / t})
print(
    f"cores: {available_cores()}, images: {len(images)} (resized to 256x256), fast mode, 1 run each"
)
pd.DataFrame(table)

## 4. GLCM + random forest vs a small CNN baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from mri_pipeline.features import image_features
from mri_pipeline.parallel import run

train_df = df[df.split == "Training"]


def glcm_matrix(paths):
    imgs = [io.load_image(p, size=256) for p in paths]
    feats = run(functools.partial(image_features, mode="fast"), imgs)
    return np.stack([f["glcm_minmax"] for f in feats])


X_train = glcm_matrix(train_df.path)
X_test = glcm_matrix(clean_test.path)
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(
    X_train, train_df.label
)
pred_rf = rf.predict(X_test)
print(classification_report(clean_test.label, pred_rf, target_names=classes, digits=3))
print(confusion_matrix(clean_test.label, pred_rf))

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)


def tensor(paths):
    arr = np.stack([io.load_image(p, size=128) for p in paths]).astype(np.float32) / 255.0
    return torch.from_numpy(arr[:, None])


xtr, ytr = tensor(train_df.path), torch.tensor(train_df.label.to_numpy())
xte, yte = tensor(clean_test.path), torch.tensor(clean_test.label.to_numpy())
model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(64, len(classes)),
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
for _epoch in range(10):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(xb.to(device)), yb.to(device))
        loss.backward()
        opt.step()
model.eval()
with torch.no_grad():
    pred_cnn = torch.cat(
        [model(xb.to(device)).argmax(1).cpu() for xb in torch.split(xte, 256)]
    ).numpy()
print(classification_report(yte.numpy(), pred_cnn, target_names=classes, digits=3))
print(confusion_matrix(yte.numpy(), pred_cnn))

## 5. Write-up checklist

- Report per-class precision/recall/F1 and the confusion matrices above, with the number of clean test images.
- Say how many test images were removed as exact or near duplicates.
- One run, one split: treat differences of a few points as noise.
- Not a medical tool.